Step 1. 重新整理分析目标

Step 2. 合并 original / JPEG predictions，做 paired prediction analysis

Step 3. 提取 fake TP→FN cases

Step 4. 按模型比较 case 数量、概率变化、方向性

Step 5. 选代表性图例

Step 6. 对代表性图例做 Grad-CAM / heatmap

Step 7. 写解释：为什么 JPEG 后 fake recall 下降，为什么 DINOv3 更明显

Rachel 的分析目标：

基于 ResNet18、ViT-B/16、DINOv3-ViT-B/16 三个 baseline 的 original / JPEG q30 controlled predictions，分析 JPEG compression 对 fake image detection 的影响。

重点关注 fake images 在 original 条件下被正确识别为 fake，但在 JPEG q30 后被误判为 real 的 cases，即 fake TP→FN cases。进一步结合 Grad-CAM / attention heatmap，解释 fake recall 下降的可能原因，尤其分析为什么 DINOv3-ViT-B/16 的 fake recall drop 最大。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Cell 0. 路径与基本设置

In [ ]:
# ============================================================
# 00_paths_and_setup
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/MLP/Project5"
OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "final_outputs")

EXPERIMENTS = {
    "resnet18": {
        "display_name": "ResNet18",
        "exp_dir": "/content/drive/MyDrive/MLP/Project5/final_outputs/resnet18_baseline_expand",
    },
    "vit_b16": {
        "display_name": "ViT-B/16",
        "exp_dir": "/content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_baseline_expand_10ep",
    },
    "dinov3_vit_b16": {
        "display_name": "DINOv3-ViT-B/16",
        "exp_dir": "/content/drive/MyDrive/MLP/Project5/final_outputs/dinov3_vit_b16_baseline_expand_10ep",
    },
}

analysis_dir = os.path.join(PROJECT_ROOT, "analysis_outputs")
os.makedirs(analysis_dir, exist_ok=True)

for model_key, info in EXPERIMENTS.items():
    exp_dir = info["exp_dir"]
    eval_dir = os.path.join(exp_dir, "eval_jpeg_q30_controlled")

    print("=" * 80)
    print(model_key, "|", info["display_name"])
    print("exp_dir exists:", os.path.exists(exp_dir))
    print("eval_dir exists:", os.path.exists(eval_dir))
    print("exp_dir:", exp_dir)

    if os.path.exists(exp_dir):
        print("exp_dir files:", os.listdir(exp_dir)[:20])
    if os.path.exists(eval_dir):
        print("eval_dir files:", os.listdir(eval_dir)[:20])

In [ ]:
# ============================================================
# 01_find_prediction_files
# ============================================================

def find_file(exp_dir, candidate_relative_paths):
    for rel_path in candidate_relative_paths:
        path = os.path.join(exp_dir, rel_path)
        if os.path.exists(path):
            return path
    return None


def get_prediction_paths(exp_dir):
    original_candidates = [
        "eval_jpeg_q30_controlled/predictions_original.csv",
        "predictions_original.csv",
    ]

    jpeg_candidates = [
        "eval_jpeg_q30_controlled/predictions_jpeg_q30_controlled.csv",
        "eval_jpeg_q30_controlled/predictions_jpeg.csv",
        "predictions_jpeg_q30_controlled.csv",
        "predictions_jpeg.csv",
    ]

    orig_path = find_file(exp_dir, original_candidates)
    jpeg_path = find_file(exp_dir, jpeg_candidates)

    return orig_path, jpeg_path


for model_key, info in EXPERIMENTS.items():
    orig_path, jpeg_path = get_prediction_paths(info["exp_dir"])

    info["pred_original_path"] = orig_path
    info["pred_jpeg_path"] = jpeg_path

    print("=" * 80)
    print(model_key)
    print("original:", orig_path)
    print("jpeg:", jpeg_path)

    if orig_path is None:
        print("WARNING: original predictions not found")
    if jpeg_path is None:
        print("WARNING: jpeg predictions not found")

In [ ]:
# ============================================================
# 02_load_prediction_files
# ============================================================

predictions = {}

for model_key, info in EXPERIMENTS.items():
    orig_path = info["pred_original_path"]
    jpeg_path = info["pred_jpeg_path"]

    if orig_path is None or jpeg_path is None:
        raise FileNotFoundError(f"Missing prediction files for {model_key}")

    orig_df = pd.read_csv(orig_path)
    jpeg_df = pd.read_csv(jpeg_path)

    predictions[model_key] = {
        "original": orig_df,
        "jpeg": jpeg_df,
    }

    print("=" * 80)
    print(model_key, info["display_name"])
    print("original shape:", orig_df.shape)
    print("jpeg shape:", jpeg_df.shape)
    print("original columns:", list(orig_df.columns))
    print("jpeg columns:", list(jpeg_df.columns))

Cell 3. 合并 original / JPEG paired predictions

In [ ]:
# ============================================================
# 03_make_paired_prediction_tables_FIXED
# Use true_label_name + image_id as pair_key
# ============================================================

def classify_transition(row):
    true_name = row["true_label_name_orig"]
    pred_orig = row["pred_label_name_orig"]
    pred_jpeg = row["pred_label_name_jpeg"]

    if true_name == "fake":
        if pred_orig == "fake" and pred_jpeg == "fake":
            return "fake_TP_to_TP"
        elif pred_orig == "fake" and pred_jpeg == "real":
            return "fake_TP_to_FN"
        elif pred_orig == "real" and pred_jpeg == "fake":
            return "fake_FN_to_TP"
        elif pred_orig == "real" and pred_jpeg == "real":
            return "fake_FN_to_FN"

    elif true_name == "real":
        if pred_orig == "real" and pred_jpeg == "real":
            return "real_TN_to_TN"
        elif pred_orig == "real" and pred_jpeg == "fake":
            return "real_TN_to_FP"
        elif pred_orig == "fake" and pred_jpeg == "real":
            return "real_FP_to_TN"
        elif pred_orig == "fake" and pred_jpeg == "fake":
            return "real_FP_to_FP"

    return "unknown"


def make_paired_table(model_key, orig_df, jpeg_df):
    needed_cols = [
        "image_id",
        "filename",
        "image_path",
        "true_label_idx",
        "true_label_name",
        "pred_label_idx",
        "pred_label_name",
        "correct",
        "prob_fake",
        "prob_real",
    ]

    missing_orig = [c for c in needed_cols if c not in orig_df.columns]
    missing_jpeg = [c for c in needed_cols if c not in jpeg_df.columns]

    if missing_orig:
        raise ValueError(f"{model_key} original missing columns: {missing_orig}")
    if missing_jpeg:
        raise ValueError(f"{model_key} jpeg missing columns: {missing_jpeg}")

    orig_small = orig_df[needed_cols].copy()
    jpeg_small = jpeg_df[needed_cols].copy()

    # 关键修正：同一个 image_id 可能同时存在于 fake / real 文件夹
    # 因此必须用 true_label_name + image_id 作为配对 key
    orig_small["pair_key"] = (
        orig_small["true_label_name"].astype(str) + "_" +
        orig_small["image_id"].astype(str)
    )

    jpeg_small["pair_key"] = (
        jpeg_small["true_label_name"].astype(str) + "_" +
        jpeg_small["image_id"].astype(str)
    )

    # 检查 pair_key 是否唯一
    orig_dup = orig_small["pair_key"].duplicated().sum()
    jpeg_dup = jpeg_small["pair_key"].duplicated().sum()

    print(f"[{model_key}] original duplicated pair_key:", orig_dup)
    print(f"[{model_key}] jpeg duplicated pair_key:", jpeg_dup)

    if orig_dup > 0:
        print("Original duplicated examples:")
        display(orig_small[orig_small["pair_key"].duplicated(keep=False)].head())

    if jpeg_dup > 0:
        print("JPEG duplicated examples:")
        display(jpeg_small[jpeg_small["pair_key"].duplicated(keep=False)].head())

    orig_small = orig_small.rename(columns={
        "filename": "filename_orig",
        "image_path": "image_path_orig",
        "true_label_idx": "true_label_idx_orig",
        "true_label_name": "true_label_name_orig",
        "pred_label_idx": "pred_label_idx_orig",
        "pred_label_name": "pred_label_name_orig",
        "correct": "correct_orig",
        "prob_fake": "prob_fake_orig",
        "prob_real": "prob_real_orig",
        "image_id": "image_id_orig",
    })

    jpeg_small = jpeg_small.rename(columns={
        "filename": "filename_jpeg",
        "image_path": "image_path_jpeg",
        "true_label_idx": "true_label_idx_jpeg",
        "true_label_name": "true_label_name_jpeg",
        "pred_label_idx": "pred_label_idx_jpeg",
        "pred_label_name": "pred_label_name_jpeg",
        "correct": "correct_jpeg",
        "prob_fake": "prob_fake_jpeg",
        "prob_real": "prob_real_jpeg",
        "image_id": "image_id_jpeg",
    })

    paired = pd.merge(
        orig_small,
        jpeg_small,
        on="pair_key",
        how="inner"
    )

    paired["model_key"] = model_key

    # 为了后续方便，保留一个统一 image_id
    paired["image_id"] = paired["image_id_orig"]

    # 检查 original / JPEG 的真实类别是否一致
    mismatch = (
        paired["true_label_name_orig"] != paired["true_label_name_jpeg"]
    ).sum()

    print(f"[{model_key}] label mismatch after merge:", mismatch)

    if mismatch > 0:
        display(
            paired[
                paired["true_label_name_orig"] != paired["true_label_name_jpeg"]
            ].head()
        )
        raise ValueError(f"{model_key}: label mismatch found after merge.")

    paired["prob_fake_drop"] = paired["prob_fake_orig"] - paired["prob_fake_jpeg"]
    paired["prob_fake_change"] = paired["prob_fake_jpeg"] - paired["prob_fake_orig"]
    paired["prob_real_change"] = paired["prob_real_jpeg"] - paired["prob_real_orig"]

    paired["transition"] = paired.apply(classify_transition, axis=1)

    return paired


paired_tables = {}

for model_key, dfs in predictions.items():
    paired = make_paired_table(
        model_key,
        dfs["original"],
        dfs["jpeg"]
    )

    paired_tables[model_key] = paired

    print("=" * 80)
    print(model_key, EXPERIMENTS[model_key]["display_name"])
    print("paired shape:", paired.shape)
    print("fake count:", (paired["true_label_name_orig"] == "fake").sum())
    print("real count:", (paired["true_label_name_orig"] == "real").sum())
    print("transition counts:")
    print(paired["transition"].value_counts())
    print()

Cell 4. 保存 paired prediction tables

In [ ]:
# ============================================================
# 04_save_paired_prediction_tables
# ============================================================

for model_key, paired in paired_tables.items():
    save_path = os.path.join(
        analysis_dir,
        f"{model_key}_paired_predictions_original_vs_jpeg.csv"
    )
    paired.to_csv(save_path, index=False, encoding="utf-8-sig")
    print("Saved:", save_path)

Cell 5. 生成 transition summary 总表

In [ ]:
# ============================================================
# 05_transition_summary
# ============================================================

summary_rows = []

for model_key, paired in paired_tables.items():
    display_name = EXPERIMENTS[model_key]["display_name"]
    counts = paired["transition"].value_counts().to_dict()

    fake_subset = paired[paired["true_label_name_orig"] == "fake"]
    real_subset = paired[paired["true_label_name_orig"] == "real"]

    row = {
        "model_key": model_key,
        "model_name": display_name,
        "total_paired": len(paired),
        "fake_total": len(fake_subset),
        "real_total": len(real_subset),

        "fake_TP_to_TP": counts.get("fake_TP_to_TP", 0),
        "fake_TP_to_FN": counts.get("fake_TP_to_FN", 0),
        "fake_FN_to_TP": counts.get("fake_FN_to_TP", 0),
        "fake_FN_to_FN": counts.get("fake_FN_to_FN", 0),

        "real_TN_to_TN": counts.get("real_TN_to_TN", 0),
        "real_TN_to_FP": counts.get("real_TN_to_FP", 0),
        "real_FP_to_TN": counts.get("real_FP_to_TN", 0),
        "real_FP_to_FP": counts.get("real_FP_to_FP", 0),

        "fake_TP_to_FN_ratio_in_fake": counts.get("fake_TP_to_FN", 0) / len(fake_subset),
        "mean_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].mean(),
        "mean_prob_fake_drop_real_images": real_subset["prob_fake_drop"].mean(),
    }

    summary_rows.append(row)

transition_summary_df = pd.DataFrame(summary_rows)

save_path = os.path.join(analysis_dir, "transition_summary_original_vs_jpeg.csv")
transition_summary_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
transition_summary_df

Cell 6. 提取 fake TP→FN cases

In [ ]:
# ============================================================
# 06_extract_fake_TP_to_FN_cases
# ============================================================

fake_tp_to_fn_cases = {}

for model_key, paired in paired_tables.items():
    cases = paired[paired["transition"] == "fake_TP_to_FN"].copy()

    # 按 fake probability drop 从大到小排序
    cases = cases.sort_values("prob_fake_drop", ascending=False)

    fake_tp_to_fn_cases[model_key] = cases

    save_path = os.path.join(
        analysis_dir,
        f"{model_key}_fake_TP_to_FN_cases.csv"
    )

    cases.to_csv(save_path, index=False, encoding="utf-8-sig")

    print("=" * 80)
    print(model_key, EXPERIMENTS[model_key]["display_name"])
    print("fake TP→FN cases:", len(cases))
    print("Saved:", save_path)

Cell 7. 查看每个模型前 10 个最明显 case

In [ ]:
# ============================================================
# 07_preview_top_fake_TP_to_FN_cases
# ============================================================

display_cols = [
    "image_id",
    "filename_orig",
    "filename_jpeg",
    "true_label_name_orig",
    "pred_label_name_orig",
    "pred_label_name_jpeg",
    "prob_fake_orig",
    "prob_fake_jpeg",
    "prob_fake_drop",
    "image_path_orig",
    "image_path_jpeg",
]

for model_key, cases in fake_tp_to_fn_cases.items():
    print("=" * 100)
    print(EXPERIMENTS[model_key]["display_name"])
    print("=" * 100)

    if len(cases) == 0:
        print("No fake TP→FN cases.")
    else:
        display(cases[display_cols].head(10))

Cell 8. 生成 XAI 候选 case 表

In [ ]:
# ============================================================
# 08_select_candidate_cases_for_xai
# ============================================================

candidate_rows = []

for model_key, cases in fake_tp_to_fn_cases.items():
    display_name = EXPERIMENTS[model_key]["display_name"]

    top_cases = cases.head(10).copy()

    for rank, (_, row) in enumerate(top_cases.iterrows(), start=1):
        candidate_rows.append({
            "model_key": model_key,
            "model_name": display_name,
            "rank": rank,
            "image_id": row["image_id"],
            "filename_orig": row["filename_orig"],
            "filename_jpeg": row["filename_jpeg"],
            "image_path_orig": row["image_path_orig"],
            "image_path_jpeg": row["image_path_jpeg"],
            "prob_fake_orig": row["prob_fake_orig"],
            "prob_fake_jpeg": row["prob_fake_jpeg"],
            "prob_fake_drop": row["prob_fake_drop"],
            "transition": row["transition"],

            # 人工标注用
            "manual_category": "",
            "manual_note": "",
            "selected_for_gradcam": "",
        })

candidate_cases_df = pd.DataFrame(candidate_rows)

save_path = os.path.join(analysis_dir, "candidate_cases_for_xai.csv")
candidate_cases_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
candidate_cases_df

cell 9. 复制候选图片，方便人工查看

In [ ]:
# ============================================================
# 09_copy_candidate_images_for_manual_review
# ============================================================

import shutil

case_img_root = os.path.join(analysis_dir, "candidate_case_images")
os.makedirs(case_img_root, exist_ok=True)

for _, row in candidate_cases_df.iterrows():
    model_key = row["model_key"]
    rank = int(row["rank"])
    image_id = str(row["image_id"])

    dst_dir = os.path.join(case_img_root, model_key, f"rank_{rank:02d}_{image_id}")
    os.makedirs(dst_dir, exist_ok=True)

    src_orig = row["image_path_orig"]
    src_jpeg = row["image_path_jpeg"]

    dst_orig = os.path.join(dst_dir, f"{image_id}_original.jpg")
    dst_jpeg = os.path.join(dst_dir, f"{image_id}_jpeg_q30.jpg")

    if os.path.exists(src_orig):
        shutil.copy2(src_orig, dst_orig)
    else:
        print("Missing original:", src_orig)

    if os.path.exists(src_jpeg):
        shutil.copy2(src_jpeg, dst_jpeg)
    else:
        print("Missing jpeg:", src_jpeg)

print("Candidate images copied to:", case_img_root)

In [ ]:
# ============================================================
# Display candidate image pairs for manual review
# ============================================================

def display_case_pair(row):
    orig_path = row["image_path_orig"]
    jpeg_path = row["image_path_jpeg"]

    orig_img = Image.open(orig_path).convert("RGB")
    jpeg_img = Image.open(jpeg_path).convert("RGB")

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(orig_img)
    plt.axis("off")
    plt.title(
        f"Original\npred=fake, prob_fake={row['prob_fake_orig']:.3f}"
    )

    plt.subplot(1, 2, 2)
    plt.imshow(jpeg_img)
    plt.axis("off")
    plt.title(
        f"JPEG q30\npred=real, prob_fake={row['prob_fake_jpeg']:.3f}"
    )

    plt.suptitle(
        f"{row['model_name']} | image_id={row['image_id']} | drop={row['prob_fake_drop']:.3f}"
    )

    plt.tight_layout()
    plt.show()


for model_key in candidate_cases_df["model_key"].unique():
    print("=" * 100)
    print(model_key)
    print("=" * 100)

    subset = candidate_cases_df[candidate_cases_df["model_key"] == model_key]

    for _, row in subset.iterrows():
        display_case_pair(row)

## **Step 4 按模型比较数量、概率变化、方向性**

### **A：probability shift summary**

In [ ]:
# ============================================================
# A_probability_shift_summary
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/MLP/Project5"
analysis_dir = os.path.join(PROJECT_ROOT, "analysis_outputs")

EXPERIMENTS = {
    "resnet18": "ResNet18",
    "vit_b16": "ViT-B/16",
    "dinov3_vit_b16": "DINOv3-ViT-B/16",
}

prob_rows = []

for model_key, model_name in EXPERIMENTS.items():
    paired_path = os.path.join(
        analysis_dir,
        f"{model_key}_paired_predictions_original_vs_jpeg.csv"
    )

    paired = pd.read_csv(paired_path)

    for true_label in ["fake", "real"]:
        subset = paired[paired["true_label_name_orig"] == true_label]

        prob_rows.append({
            "model_key": model_key,
            "model_name": model_name,
            "true_label": true_label,
            "n": len(subset),
            "mean_prob_fake_orig": subset["prob_fake_orig"].mean(),
            "mean_prob_fake_jpeg": subset["prob_fake_jpeg"].mean(),
            "mean_prob_fake_drop": subset["prob_fake_drop"].mean(),
            "median_prob_fake_drop": subset["prob_fake_drop"].median(),
            "std_prob_fake_drop": subset["prob_fake_drop"].std(),
        })

prob_shift_df = pd.DataFrame(prob_rows)

save_path = os.path.join(analysis_dir, "probability_shift_summary_original_vs_jpeg.csv")
prob_shift_df.to_csv(save_path, index=False, encoding="utf-8-sig")

prob_shift_df

### **B：file extension summary**

In [ ]:
# ============================================================
# B_file_extension_summary_for_fake_TP_to_FN
# ============================================================

import os
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/MLP/Project5"
analysis_dir = os.path.join(PROJECT_ROOT, "analysis_outputs")

EXPERIMENTS = {
    "resnet18": "ResNet18",
    "vit_b16": "ViT-B/16",
    "dinov3_vit_b16": "DINOv3-ViT-B/16",
}

ext_rows = []

for model_key, model_name in EXPERIMENTS.items():
    case_path = os.path.join(
        analysis_dir,
        f"{model_key}_fake_TP_to_FN_cases.csv"
    )

    cases = pd.read_csv(case_path)

    cases["orig_ext"] = cases["filename_orig"].apply(
        lambda x: os.path.splitext(str(x))[1].lower()
    )
    cases["jpeg_ext"] = cases["filename_jpeg"].apply(
        lambda x: os.path.splitext(str(x))[1].lower()
    )

    ext_count = cases.groupby(["orig_ext", "jpeg_ext"]).size().reset_index(name="count")
    ext_count["model_key"] = model_key
    ext_count["model_name"] = model_name
    ext_count["total_fake_TP_to_FN"] = len(cases)
    ext_count["ratio"] = ext_count["count"] / len(cases)

    ext_rows.append(ext_count)

ext_summary_df = pd.concat(ext_rows, ignore_index=True)

save_path = os.path.join(analysis_dir, "fake_TP_to_FN_file_extension_summary.csv")
ext_summary_df.to_csv(save_path, index=False, encoding="utf-8-sig")

ext_summary_df

### **C：transition summary 图表**

In [ ]:
# ============================================================
# C_plot_fake_TP_to_FN_comparison
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = "/content/drive/MyDrive/MLP/Project5"
analysis_dir = os.path.join(PROJECT_ROOT, "analysis_outputs")

summary_path = os.path.join(analysis_dir, "transition_summary_original_vs_jpeg.csv")
transition_summary_df = pd.read_csv(summary_path)

fig_dir = os.path.join(analysis_dir, "figures")
os.makedirs(fig_dir, exist_ok=True)

plt.figure(figsize=(7, 5))
plt.bar(
    transition_summary_df["model_name"],
    transition_summary_df["fake_TP_to_FN"]
)
plt.ylabel("Number of fake TP→FN cases")
plt.xlabel("Model")
plt.title("Fake images flipped from correct fake to real after JPEG q30")
plt.xticks(rotation=20)
plt.tight_layout()

save_path = os.path.join(fig_dir, "fake_TP_to_FN_count_comparison.png")
plt.savefig(save_path, dpi=300)
plt.show()

print("Saved:", save_path)

In [ ]:
# ============================================================
# C2_plot_fake_TP_to_FN_ratio
# ============================================================

transition_summary_df["fake_TP_to_FN_ratio_percent"] = (
    transition_summary_df["fake_TP_to_FN"] / transition_summary_df["fake_total"] * 100
)

plt.figure(figsize=(7, 5))
plt.bar(
    transition_summary_df["model_name"],
    transition_summary_df["fake_TP_to_FN_ratio_percent"]
)
plt.ylabel("Fake TP→FN ratio among fake images (%)")
plt.xlabel("Model")
plt.title("Ratio of fake images misclassified as real after JPEG q30")
plt.xticks(rotation=20)
plt.tight_layout()

save_path = os.path.join(fig_dir, "fake_TP_to_FN_ratio_comparison.png")
plt.savefig(save_path, dpi=300)
plt.show()

print("Saved:", save_path)

transition_summary_df[
    ["model_name", "fake_total", "fake_TP_to_FN", "fake_TP_to_FN_ratio_percent"]
]

### **D：整理一个小结表**

In [ ]:
# ============================================================
# D_make_analysis_summary_table
# ============================================================

prob_path = os.path.join(analysis_dir, "probability_shift_summary_original_vs_jpeg.csv")
prob_shift_df = pd.read_csv(prob_path)

fake_prob_df = prob_shift_df[prob_shift_df["true_label"] == "fake"].copy()

analysis_summary = transition_summary_df.merge(
    fake_prob_df[
        [
            "model_key",
            "mean_prob_fake_orig",
            "mean_prob_fake_jpeg",
            "mean_prob_fake_drop",
            "median_prob_fake_drop",
        ]
    ],
    on="model_key",
    how="left"
)

analysis_summary = analysis_summary[
    [
        "model_key",
        "model_name",
        "fake_total",
        "fake_TP_to_FN",
        "fake_TP_to_FN_ratio_percent",
        "fake_FN_to_TP",
        "mean_prob_fake_orig",
        "mean_prob_fake_jpeg",
        "mean_prob_fake_drop",
        "median_prob_fake_drop",
    ]
]

save_path = os.path.join(analysis_dir, "baseline_paired_analysis_summary.csv")
analysis_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
analysis_summary